In [2]:
# Block 1: Load Data for Feature Engineering
import pandas as pd
import numpy as np

print(" Starting Feature Engineering...\n")

df_emp = pd.read_csv("D:\\HR Project\\HR data\\curated\\Dim_Employees_Curated.csv")
df_att = pd.read_csv("D:\\HR Project\\HR data\\curated\\Fact_Attendance_Curated.csv")
df_comp = pd.read_csv("D:\\HR Project\\HR data\\curated\\Fact_Compensation_Curated.csv")

# Convert dates
df_emp['birth_date'] = pd.to_datetime(df_emp['birth_date'])
df_emp['hire_date'] = pd.to_datetime(df_emp['hire_date'])
df_emp['exit_date'] = pd.to_datetime(df_emp['exit_date'])

print(" Data Loaded Successfully.")

 Starting Feature Engineering...

 Data Loaded Successfully.


In [3]:
# Block 2: Employee Features (Age & Tenure)
print(" Engineering Features for Dim_Employees...")

# 1. Calculate Age (in years)
# Using a fixed reference date (e.g., end of analysis period) for consistency
reference_date = pd.Timestamp('2026-01-01')
df_emp['age_years'] = ((reference_date - df_emp['birth_date']).dt.days / 365.25).round(1)

# 2. Calculate Tenure (مدة الخدمة)
# If employee resigned, tenure is (exit_date - hire_date)
# If still active, tenure is (reference_date - hire_date)
df_emp['end_date_for_tenure'] = df_emp['exit_date'].fillna(reference_date)
df_emp['tenure_years'] = ((df_emp['end_date_for_tenure'] - df_emp['hire_date']).dt.days / 365.25).round(1)

# Clean up temporary column
df_emp.drop(columns=['end_date_for_tenure'], inplace=True)

print(" Age and Tenure calculated successfully.")
display(df_emp[['employee_id', 'age_years', 'tenure_years', 'attrition_flag']].head(3))

 Engineering Features for Dim_Employees...
 Age and Tenure calculated successfully.


,employee_id,age_years,tenure_years,attrition_flag
0,1,27.4,3.2,No
1,2,48.6,5.7,No
2,3,29.4,4.9,Yes


In [5]:
# Block 3: Fact Tables Features (Overtime Ratio)
print("\n Engineering Features for Fact_Attendance...")

# Calculate Overtime Ratio (Avoid division by zero)
# Example: 20 hours overtime / 200 total hours = 10% overtime ratio
df_att['overtime_ratio_pct'] = np.where(
    df_att['total_working_hours'] > 0, 
    (df_att['overtime_hours'] / df_att['total_working_hours']) * 100, 
    0
).round(2)

print(" Overtime Ratio calculated successfully.")
display(df_att[['employee_id', 'total_working_hours', 'overtime_hours', 'overtime_ratio_pct']].head(3))


 Engineering Features for Fact_Attendance...
 Overtime Ratio calculated successfully.


,employee_id,total_working_hours,overtime_hours,overtime_ratio_pct
0,1,159,28,17.61
1,1,168,35,20.83
2,1,199,21,10.55


In [7]:
# Block 4: Exporting the FINAL Analysis-Ready Data
print("\n Exporting the FINAL Layer...")

df_emp.to_csv("Dim_Employees_Final.csv", index=False)
df_att.to_csv("Fact_Attendance_Final.csv", index=False)
# Other curated files don't need changes, but let's copy them to "Final" for consistency
import shutil
shutil.copy("D:\HR Project\HR data\curated\Dim_Job_Roles_Curated.csv", "Dim_Job_Roles_Final.csv")
shutil.copy("D:\HR Project\HR data\curated\Fact_Compensation_Curated.csv", "Fact_Compensation_Final.csv")
shutil.copy("D:\HR Project\HR data\curated\Fact_Engagement_Curated.csv", "Fact_Engagement_Final.csv")
shutil.copy("D:\HR Project\HR data\curated\Fact_Performance_Curated.csv", "Fact_Performance_Final.csv")


 Exporting the FINAL Layer...


'Fact_Performance_Final.csv'